In [0]:
# Read the values passed from the job UI FOR DEPLOYMENT
import os
import json

raw = dbutils.jobs.taskValues.get("ingest_customer_data", "metadata", None)
if raw:
    print("Running in Job?", True)
    print(f"raw ---> {raw}")

    # get data from the json output of the job ingest_customers_data
    meta = json.loads(raw)
    catalog          = meta.get("catalog")
    schema           = meta.get("schema")
    data_ingestion_volume = meta.get("data_ingestion_volume")
    customers_table  = meta.get("customers_table")
    orders_data      = meta.get("orders_data")

    # catalog = dbutils.widgets.get("catalog")
    # schema  = dbutils.widgets.get("schema")
    delta_table_volume = dbutils.widgets.get("delta_table_volume")
    # data_ingestion_volume = dbutils.widgets.get("data_ingestion_volume")

    # delta table related paths
    delta_table_schema_store = dbutils.widgets.get("delta_table_schema_store")
    delta_table_checkpoints = dbutils.widgets.get("delta_table_checkpoints")
    delta_table_path = dbutils.widgets.get("delta_table_path")

    # data ingestion raw file related paths
    # orders_data = dbutils.widgets.get("orders_data")

    # customer_table_name
    orders_table = dbutils.widgets.get("orders_table")

    # for deployment
    orders_delta_table_schema_path = dbutils.widgets.get("orders_delta_table_schema_path")
    orders_delta_table_checkpoint_path = dbutils.widgets.get("orders_delta_table_checkpoint_path")
    orders_delta_table_location = dbutils.widgets.get("orders_delta_table_location")

    # job orchestration related configurations
    second_job_id = dbutils.widgets.get("second_job_id")
    third_job_id = dbutils.widgets.get("third_job_id")
    databricks_host = dbutils.widgets.get("databricks_host")
    job_orchestration_token = dbutils.widgets.get("job_orchestration_token")
else:
    catalog = "job_orchestration"
    schema  = "default"
    delta_table_volume = "orders_volume"
    data_ingestion_volume = "job_orchestration_volume"

    # data table related paths
    delta_table_schema_store = "/Volumes/job_orchestration/default/orders_volume/schema_store"
    delta_table_checkpoints = "/Volumes/job_orchestration/default/orders_volume/checkpoints"
    delta_table_path = "/Volumes/job_orchestration/default/orders_volume/orders_table"

    # data ingestion related raw files related paths
    # customers_data = "/Volumes/job_orchestration/default/job_orchestration_volume/ingest_data/customers_data/"
    orders_data = "/Volumes/job_orchestration/default/job_orchestration_volume/ingest_data/orders_data/"
    # sales_data = "/Volumes/job_orchestration/default/job_orchestration_volume/ingest_data/sales_data/"

    # customer_table_name
    orders_table="orders_table"

    # for testing
    orders_delta_table_schema_path          = "/Volumes/job_orchestration/default/orders_volume/schema_store/"
    orders_delta_table_checkpoint_path      = "/Volumes/job_orchestration/default/orders_volume/checkpoints/"
    # If I pass this customer_delta_table path like this then the table created will not be registered in the unity catalog and as a result it will not be accesible by using sql commands
    # customer_delta_table_location       = "/Volumes/job_orchestration/default/orders_volume/orders_table/"
    # If i pass this customer_delta_table path like this then the table created will be registered in the unity catalog and as a result it will be accesible by suing sql commands
    orders_delta_table_location       = "job_orchestration.default.orders_table"

print("Catalog:", catalog)
print("Schema:", schema)

spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{delta_table_volume};")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{data_ingestion_volume};")

In [0]:
spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema}.{orders_table} USING DELTA;
""")

In [0]:
from pyspark.sql.utils import AnalysisException

# Check if csv file in customer_data location exists or not
def path_has_files(path):
    try:
        files = dbutils.fs.ls(path)
        # Filter out only real data files (csv, parquet, etc.)
        data_files = [f for f in files if f.name.lower().endswith(".csv")]
        return len(data_files) > 0
    except Exception:
        # Directory does NOT exist
        return False
data_ingested = False
if path_has_files(orders_data):
    print("✔ CSV files found — starting Auto Loader ingestion...")
    df = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.schemaLocation", orders_delta_table_schema_path)
            .load(orders_data)
    )

    # This code generates the table that is registered in the unity catalog hence it can be accessed via sql commands 
    (
        df.writeStream
        .option("checkpointLocation", orders_delta_table_checkpoint_path)
        .option("mergeSchema", "true")
        .outputMode("append")
        .trigger(availableNow=True)
        # .table("job_orchestration.default.customers_table")
        .table(orders_delta_table_location)
    )
    data_ingested = True
else:
    print("✘ No CSV files found — skipping ingestion.")
    data_ingested = False
    pass


In [0]:
# Job orchestration logic
import json
from pyspark.sql import Row
import requests

# Now trigger Ingest_sales_data using databricks api from this task in job 1 and send the relavant required data to job 2 which is essential to run data-ingestion logic in job 2
if data_ingested:
    resp = requests.post(
        databricks_host+"/api/2.1/jobs/run-now",
        headers={"Authorization": f"Bearer {job_orchestration_token}"},
        json={"job_id": second_job_id, "notebook_params": {
            "status": "success",
            "catalog": catalog,
            "schema": schema,
            "data_ingestion_volume": data_ingestion_volume,
            "databricks_host":databricks_host,
            "job_orchestration_token":job_orchestration_token,
        }},
    )

    # Debugging
    # since job orchestration is not working properly, I am inspecting the response I get from this api call that I made to trigger job-2's execution programmatically.
    print(f"response for job_id : {second_job_id}")
    print(resp.status_code)
    print(resp.text)        # should include run_id or error details
    # If status 200, parse JSON:
    print(resp.json())
else:
    print(f"No data ingested triggering job : {second_job_id} skipped")

if data_ingested:
    resp = requests.post(
        databricks_host+"/api/2.1/jobs/run-now",
        headers={"Authorization": f"Bearer {job_orchestration_token}"},
        json={"job_id": third_job_id, 
            "task_key":"join_customer_end_sales_data", # task_key is required for multi-task jobs
            "notebook_params": {
                    "status": "success",
                    "catalog": catalog,
                    "schema": schema,
                    "data_ingestion_volume": data_ingestion_volume,
                    "databricks_host":databricks_host,
                    "job_orchestration_token":job_orchestration_token
                }},
    )

    # Debugging
    # since job orchestration is not working properly, I am inspecting the response I get from this api call that I made to trigger job-2's execution programmatically.
    print(f"response for job_id : {third_job_id}")
    print(resp.status_code)
    print(resp.text)        # should include run_id or error details
    # If status 200, parse JSON:
    print(resp.json())
else:
    print(f"No data ingested triggering job : {third_job_id} skipped")

<Response [200]>